In [ ]:
# Check pinned package versions.
import importlib.metadata as meta

REQUIRED = {"numpy": "1.26.4", "scipy": "1.13.1", "scikit-learn": "1.6.1"}
mismatch = {p: (v, meta.version(p)) for p, v in REQUIRED.items()
            if meta.version(p) != v}
if mismatch:
    raise RuntimeError(
        "Version mismatch: "
        + ", ".join(f"{p} needs {w}, found {g}" for p, (w, g) in mismatch.items())
        + "\npip install " + " ".join(f"{p}=={v}" for p, v in REQUIRED.items()))

In [ ]:
import os
import time
import urllib.request

from bpcal import (build_adjacency, calibrate, cluster, compatibility_kernel,
                   edge_discrepancy, evaluate, load_npz, local_scale,
                   preprocess, row_stochastic)

In [ ]:
# Hyperparameters.
k_nn  = 15      # spatial graph degree
gamma = 2.0     # kernel bandwidth scaling
eta   = 0.3     # absorption ratio
steps = 10      # calibration steps
n_pcs = 30      # PCA dimension

In [ ]:
# Load osmFISH.
repo = "SiyuanZhou1024/BPCal"
data = "osmfish.npz"

if not os.path.exists(data):
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{repo}/main/Data_and_Demo/{data}", data)

counts, coords, truth = load_npz(data)
n_domains = len(set(truth))

print(f"{len(coords)} spots, {counts.shape[1]} genes, {n_domains} domains")

In [ ]:
# Normalise, log-transform and reduce to the initial embedding.
embedding = preprocess(counts, n_pcs)

In [ ]:
# Build the spatial graph, then measure edge discrepancy and local scale.
t0 = time.perf_counter()
adjacency = build_adjacency(coords, k_nn)
discrepancy = edge_discrepancy(embedding, adjacency)
scale = local_scale(discrepancy, adjacency)
t_graph = time.perf_counter() - t0

In [ ]:
# Build the bilateral compatibility kernel and calibrate.
t0 = time.perf_counter()
kernel = compatibility_kernel(discrepancy, scale, adjacency, gamma)
calibrated = calibrate(embedding, row_stochastic(kernel), eta, steps)
t_calibrate = time.perf_counter() - t0

In [ ]:
# Cluster and evaluate.
labels = cluster(calibrated, n_domains)
scores = evaluate(labels, truth)

print(f"ARI     = {scores['ARI']:.2f}")
print(f"NMI     = {scores['NMI']:.2f}")
print(f"runtime = {t_graph + t_calibrate:.2f} s")